In [1]:
import numpy as np
import numpy.random as rd
import json
from pathlib import Path

PROJECT_ROOT = Path("/Users/xueqingliu/Harvard University Dropbox/Liu Xueqing/ADAPR-MRT-Testbed")
PARAMS_DIR = PROJECT_ROOT / "env_para_vanilla"

In [2]:
class EnvConfig:
    """Load fitted environment parameters for one participant.

    Parameters match those saved by 2.1_fit_vanilla_testbed (cells 8-9).
    Coefficient vectors (theta_*) and bootstrap residuals (resid_*) are read
    from ``params_env_{userid}.json``; variable limits come from
    ``std_params.json``.
    """

    def __init__(self, userid, params_dir=PARAMS_DIR):
        self.userid = userid
        self.K = 2       # decision points per day (morning / afternoon)
        self.W = 7       # days per week
        self.nweek = 11  # weeks per episode (weeks 1-11 after preprocessing)
        self.D = self.nweek * self.W

        # --- limits / scales ---
        with open(Path(params_dir) / "std_params.json") as f:
            std = json.load(f)
        self.limits_fourSC      = std["4hour_step_count_limit"]
        self.limits_pageview    = std["HourlyPageviewCount_limit"]
        self.limits_CAE         = std["CAE_avg_limit"]
        self.limits_CAE_short   = std["CAE_short_avg_limit"]
        self.limits_antic         = std["anticipated_affect_yesterday_limit"]
        self.limits_fitbitwearing = [0.0, 1.0]
        self.limits_dailysurvey   = [0.0, 1.0]
        self.limits_perceived_utility = [0.0, 10.0]
        self.limits_week_present  = [0.0, 1.0]

        # --- user-specific parameters ---
        with open(Path(params_dir) / f"params_env_{userid}.json") as f:
            p = json.load(f)

        # regression coefficients
        self.theta_fourSC           = np.array(p["theta_fourSC"])            # (35,)
        self.theta_pageview         = np.array(p["theta_pageview"])          # (23,) 17 orig + 6 query
        self.theta_fitbitwearing    = np.array(p["theta_fitbitwearing"])     # (26,) 21 orig + 5 query
        self.theta_dailysurvey      = np.array(p["theta_dailysurvey"])      # (21,)
        self.theta_eodcomplete      = np.array(p.get("theta_eodcomplete", []))  # (5,) query for dailysurvey
        self.theta_CAE              = np.array(p["theta_CAE"])              # (24,)
        self.theta_perceived_utility = np.array(p["theta_perceived_utility"])  # (24,)
        self.theta_week_present     = np.array(p["theta_week_present"])     # (2,)
        self.theta_CAE_short        = np.array(p["theta_CAE_short_avg"])    # (2,)
        self.theta_antic            = np.array(p["theta_antic"])            # (25,)

        # residuals (bootstrap noise source)
        self.resid_fourSC       = np.array(p["resid_fourSC"])       # (154,) may contain NaN
        self.resid_antic        = np.array(p["resid_antic"])        # daily, obs only
        self.resid_pageview     = np.array(p["resid_pageview"])     # (77,)
        self.resid_fitbitwearing = np.array(p["resid_fitbitwearing"])  # (77,)
        self.resid_dailysurvey  = np.array(p["resid_dailysurvey"])  # (77,)
        self.resid_CAE          = np.array(p["resid_CAE"])          # (~9,)
        self.resid_perceived_utility = np.array(p["resid_perceived_utility"])  # (~9,)
        self.resid_week_present = np.array(p["resid_week_present"]) # (11,)
        self.resid_CAE_short    = np.array(p["resid_CAE_short_avg"])  # (11,) may contain NaN

        # observed-only residuals (NaN removed) for fallback sampling
        self.resid_obs_fourSC    = self.resid_fourSC[~np.isnan(self.resid_fourSC)]
        self.resid_obs_CAE_short = self.resid_CAE_short[~np.isnan(self.resid_CAE_short)]


# quick smoke test
_cfg = EnvConfig(118)
print(f"Loaded user {_cfg.userid}: theta_fourSC {_cfg.theta_fourSC.shape}, "
      f"resid_fourSC {_cfg.resid_fourSC.shape}")

Loaded user 118: theta_fourSC (35,), resid_fourSC (154,)


In [3]:
class Env:
    """Generative environment for the vanilla testbed.

    Mirrors the structure of env_testbed.py in the STE6 reference code,
    adapted for the ADAPR vanilla testbed's fitted models.

    Each ``gen_*_mean`` method builds the predictor vector that matches
    the design matrix used in ``2.1_fit_vanilla_testbed`` and returns the
    conditional mean.  The corresponding ``gen_*`` method adds bootstrap
    noise and clips the result to the variable's feasible range.

    """

    def __init__(self, env_config: EnvConfig, noise="sequential"):
        assert noise in ("sequential", "random")
        self.noise = noise
        self.cfg = env_config
        self.K = env_config.K
        self.W = env_config.W

    # ----- noise sampling (matches env_testbed.py) -----

    def _sample_noise(self, resid, idx, obs_resid=None):
        if obs_resid is None:
            obs_resid = resid[~np.isnan(resid)]
        if len(obs_resid) == 0:
            return 0.0
        if self.noise == "sequential":
            val = resid[idx % len(resid)]
            return val if not np.isnan(val) else rd.choice(obs_resid)
        return rd.choice(obs_resid)

    # ================================================================
    #  Decision-level mediators  (K=2 per day)
    # ================================================================

    # ---- M^Y: 4-hour step count (fourSC) ----
    # Predictor order (35 = 19 main + 16 WS interactions):
    #   Main: [1, fourSC_lag1, yesterday_step, 7d_step_avg, prior2h_step,
    #          Prev7DaysRPA, recent_burden, 7d_pageview, past7d_wearing,
    #          yest_planning, yest_salience, interact7d_walk, interact7d_sal,
    #          antic_affect_yest, is_weekend, dow_norm, decision_time,
    #          CAE_lastweek, PU_lastweek]
    #   Trt:  WS * [1, yesterday_step, prior2h_step, recent_burden,
    #               7d_pageview, past7d_wearing, yest_planning, yest_salience,
    #               interact7d_walk, interact7d_sal, antic_affect_yest,
    #               is_weekend, dow_norm, decision_time,
    #               CAE_lastweek, PU_lastweek]

    def gen_fourSC_mean(self, s, Ah):
        X = np.array([
            1.0, s["fourSC_lag1"], s["yesterday_step"], s["seven_day_step_avg"],
            s["prior2hour_step"], s["Previous7DaysRPA"], s["recent_burden"],
            s["seven_day_pageview"], s["past7days_daywearing"],
            s["yesterday_planning_prompt"], s["yesterday_SalienceMessage"],
            s["Interacted_7d_walk"], s["Interacted_7d_salience"],
            s["anticipated_affect_yesterday"], s["is_weekend"], s["dow_norm"],
            s["decision_time"], s["CAE_avg_lastweek"], s["perceived_utility_lastweek"],
            Ah,
            Ah * s["yesterday_step"],       Ah * s["prior2hour_step"],
            Ah * s["recent_burden"],        Ah * s["seven_day_pageview"],
            Ah * s["past7days_daywearing"], Ah * s["yesterday_planning_prompt"],
            Ah * s["yesterday_SalienceMessage"], Ah * s["Interacted_7d_walk"],
            Ah * s["Interacted_7d_salience"], Ah * s["anticipated_affect_yesterday"],
            Ah * s["is_weekend"],           Ah * s["dow_norm"],
            Ah * s["decision_time"],        Ah * s["CAE_avg_lastweek"],
            Ah * s["perceived_utility_lastweek"],
        ])
        return self.cfg.theta_fourSC @ X

    def gen_fourSC(self, s, Ah, step_idx):
        mean = self.gen_fourSC_mean(s, Ah)
        noise = self._sample_noise(
            self.cfg.resid_fourSC, step_idx, self.cfg.resid_obs_fourSC
        )
        return np.clip(mean + noise, *self.cfg.limits_fourSC)

    # ---- M^E: hourly pageview ----
    # Original (17): [1, pv_lag1, fitbit_morn, daily_present, recent_burden,
    #                  interact7d_walk, interact7d_sal, dow_norm, decision_time,
    #                  CAE_lastweek, PU_lastweek,
    #                  WS, WS*recent_burden, WS*dow_norm, WS*decision_time,
    #                  WS*CAE_lastweek, WS*PU_lastweek]
    # Query  (6):  Iw * [1, recent_burden, dow_norm, decision_time,
    #                     CAE_lastweek, PU_lastweek]

    def gen_pageview_mean(self, s, Ah, Iw=0):
        X_orig = np.array([
            1.0, s["hourly_pageview_lag1"], s["fitbitwearing_morning"],
            s["daily_present"], s["recent_burden"],
            s["Interacted_7d_walk"], s["Interacted_7d_salience"],
            s["dow_norm"], s["decision_time"],
            s["CAE_avg_lastweek"], s["perceived_utility_lastweek"],
            Ah,
            Ah * s["recent_burden"],        Ah * s["dow_norm"],
            Ah * s["decision_time"],        Ah * s["CAE_avg_lastweek"],
            Ah * s["perceived_utility_lastweek"],
        ])
        mean = self.cfg.theta_pageview[:17] @ X_orig
        if Iw != 0 and len(self.cfg.theta_pageview) > 17:
            X_q = np.array([
                1.0, s["recent_burden"], s["dow_norm"], s["decision_time"],
                s["CAE_avg_lastweek"], s["perceived_utility_lastweek"],
            ])
            mean += Iw * self.cfg.theta_pageview[17:23] @ X_q
        return mean

    def gen_pageview(self, s, Ah, Iw, step_idx):
        mean = self.gen_pageview_mean(s, Ah, Iw)
        noise = self._sample_noise(self.cfg.resid_pageview, step_idx)
        return np.clip(mean + noise, *self.cfg.limits_pageview)

    # ================================================================
    #  Daily mediators  (one per day, using morning-row predictor layout)
    # ================================================================

    # ---- M^Y_daily: anticipated affect ----
    # Predictor order (25 = 9 main + 16 WS morning/afternoon interactions):
    #   Main: [1, antic_lag1, yesterday_step, recorded_phys_lag1,
    #          yest_planning, yest_salience, dow_norm_lag1,
    #          CAE_lastweek, PU_lastweek]
    #   Trt:  [ws_m, ws_a,
    #          ws_m*yest_step, ws_a*yest_step,
    #          ws_m*rec_phys,  ws_a*rec_phys,
    #          ws_m*plan,      ws_a*plan,
    #          ws_m*sal,       ws_a*sal,
    #          ws_m*dow,       ws_a*dow,
    #          ws_m*CAE,       ws_a*CAE,
    #          ws_m*PU,        ws_a*PU]
    # Response: anticipated_affect_yesterday_norm (yesterday's antic. affect)
    # Fitted on morning rows only (decision_time == 0).

    def gen_antic_mean(self, s, ws_morning, ws_afternoon):
        ys  = s["yesterday_step"]
        rpa = s["recorded_physical_activity_lag1"]
        pl  = s["yesterday_planning_prompt"]
        sal = s["yesterday_SalienceMessage"]
        dow = s["dow_norm_lag1"]
        cae = s["CAE_avg_lastweek"]
        pu  = s["perceived_utility_lastweek"]
        X = np.array([
            1.0, s["anticipated_affect_lag1"], ys, rpa, pl, sal, dow, cae, pu,
            ws_morning, ws_afternoon,
            ws_morning * ys,  ws_afternoon * ys,
            ws_morning * rpa, ws_afternoon * rpa,
            ws_morning * pl,  ws_afternoon * pl,
            ws_morning * sal, ws_afternoon * sal,
            ws_morning * dow, ws_afternoon * dow,
            ws_morning * cae, ws_afternoon * cae,
            ws_morning * pu,  ws_afternoon * pu,
        ])
        return self.cfg.theta_antic @ X

    def gen_antic(self, s, ws_morning, ws_afternoon, day_idx):
        mean = self.gen_antic_mean(s, ws_morning, ws_afternoon)
        noise = self._sample_noise(self.cfg.resid_antic, day_idx)
        return np.clip(mean + noise, *self.cfg.limits_antic)

    # ---- M^E_daily: fitbit wearing (morning) ----
    # Original (21): [1, yest_fitbit, pv_morn_yest, pv_aftn_yest,
    #                  daily_present*, recent_burden,
    #                  interact7d_walk, interact7d_sal, dow_norm_lag1,
    #                  CAE_lastweek, PU_lastweek,
    #                  ws_morn, ws_aftn,
    #                  ws_morn*rb, ws_aftn*rb,
    #                  ws_morn*dow, ws_aftn*dow,
    #                  ws_morn*CAE, ws_aftn*CAE,
    #                  ws_morn*PU,  ws_aftn*PU]
    # * daily_present uses yesterday's value (avoids circularity)
    # Query (5): Iw * [1, rb, dow_lag1, CAE_lastweek, PU_lastweek]

    def gen_fitbitwearing_mean(self, s, ws_morning, ws_afternoon, Iw=0):
        rb  = s["recent_burden"]
        dow = s["dow_norm_lag1"]
        cae = s["CAE_avg_lastweek"]
        pu  = s["perceived_utility_lastweek"]
        X_orig = np.array([
            1.0, s["yesterday_fitbitwearing"],
            s["yesterday_pageview_morning"], s["yesterday_pageview_afternoon"],
            s["yesterday_present"], rb,
            s["Interacted_7d_walk"], s["Interacted_7d_salience"],
            dow, cae, pu,
            ws_morning, ws_afternoon,
            ws_morning * rb,  ws_afternoon * rb,
            ws_morning * dow, ws_afternoon * dow,
            ws_morning * cae, ws_afternoon * cae,
            ws_morning * pu,  ws_afternoon * pu,
        ])
        mean = self.cfg.theta_fitbitwearing[:21] @ X_orig
        if Iw != 0 and len(self.cfg.theta_fitbitwearing) > 21:
            X_q = np.array([1.0, rb, dow, cae, pu])
            mean += Iw * self.cfg.theta_fitbitwearing[21:26] @ X_q
        return mean

    def gen_fitbitwearing(self, s, ws_morning, ws_afternoon, Iw, day_idx):
        mean = self.gen_fitbitwearing_mean(s, ws_morning, ws_afternoon, Iw)
        noise = self._sample_noise(self.cfg.resid_fitbitwearing, day_idx)
        return np.clip(mean + noise, *self.cfg.limits_fitbitwearing)

    # ---- M^E_daily: end-of-day survey completion ----
    # Original (21): [1, yest_present, fitbit_morning_today,
    #                  pv_morn_yest, pv_aftn_yest, recent_burden,
    #                  interact7d_walk, interact7d_sal, dow_norm_lag1,
    #                  CAE_lastweek, PU_lastweek,
    #                  ws_morn, ws_aftn,
    #                  ws_morn*rb, ws_aftn*rb,
    #                  ws_morn*dow, ws_aftn*dow,
    #                  ws_morn*CAE, ws_aftn*CAE,
    #                  ws_morn*PU,  ws_aftn*PU]
    # Query (5): Iw * [1, rb, dow_lag1, CAE_lastweek, PU_lastweek]

    def gen_dailysurvey_mean(self, s, ws_morning, ws_afternoon, Iw=0):
        rb  = s["recent_burden"]
        dow = s["dow_norm_lag1"]
        cae = s["CAE_avg_lastweek"]
        pu  = s["perceived_utility_lastweek"]
        X_orig = np.array([
            1.0, s["yesterday_present"], s["fitbitwearing_morning"],
            s["yesterday_pageview_morning"], s["yesterday_pageview_afternoon"],
            rb,
            s["Interacted_7d_walk"], s["Interacted_7d_salience"],
            dow, cae, pu,
            ws_morning, ws_afternoon,
            ws_morning * rb,  ws_afternoon * rb,
            ws_morning * dow, ws_afternoon * dow,
            ws_morning * cae, ws_afternoon * cae,
            ws_morning * pu,  ws_afternoon * pu,
        ])
        mean = self.cfg.theta_dailysurvey @ X_orig
        if Iw != 0 and len(self.cfg.theta_eodcomplete) > 0:
            X_q = np.array([1.0, rb, dow, cae, pu])
            mean += Iw * self.cfg.theta_eodcomplete @ X_q
        return mean

    def gen_dailysurvey(self, s, ws_morning, ws_afternoon, Iw, day_idx):
        mean = self.gen_dailysurvey_mean(s, ws_morning, ws_afternoon, Iw)
        noise = self._sample_noise(self.cfg.resid_dailysurvey, day_idx)
        return np.clip(mean + noise, *self.cfg.limits_dailysurvey)

    # ================================================================
    #  Weekly outcomes
    # ================================================================

    # ---- Y_w: CAE (affective association) ----
    # Predictors (24): [1, CAE_lastweek, week_norm,
    #                    foursc_wk (14 decision slots),
    #                    antic_wk  (7 daily values)]

    def gen_CAE_mean(self, CAE_lastweek, week_norm, foursc_wk, antic_wk):
        X = np.concatenate([
            [1.0, CAE_lastweek, week_norm],
            foursc_wk.ravel(),   # 14
            antic_wk.ravel(),    #  7
        ])
        return self.cfg.theta_CAE @ X

    def gen_CAE(self, CAE_lastweek, week_norm, foursc_wk, antic_wk, week_idx):
        mean = self.gen_CAE_mean(CAE_lastweek, week_norm, foursc_wk, antic_wk)
        noise = self._sample_noise(self.cfg.resid_CAE, week_idx)
        return np.clip(mean + noise, *self.cfg.limits_CAE)

    # ---- E_w: perceived utility ----
    # Predictors (24): [1, PU_lastweek, week_norm,
    #                    pw_wk (7), dw_wk (7), dp_wk (7)]

    def gen_perceived_utility_mean(self, pu_lastweek, week_norm,
                                   pw_wk, dw_wk, dp_wk):
        X = np.concatenate([
            [1.0, pu_lastweek, week_norm],
            pw_wk.ravel(),   # 7
            dw_wk.ravel(),   # 7
            dp_wk.ravel(),   # 7
        ])
        return self.cfg.theta_perceived_utility @ X

    def gen_perceived_utility(self, pu_lastweek, week_norm,
                              pw_wk, dw_wk, dp_wk, week_idx):
        mean = self.gen_perceived_utility_mean(
            pu_lastweek, week_norm, pw_wk, dw_wk, dp_wk
        )
        noise = self._sample_noise(self.cfg.resid_perceived_utility, week_idx)
        return np.clip(mean + noise, *self.cfg.limits_perceived_utility)

    # ---- Emissions ----

    def gen_week_present_mean(self, perceived_utility):
        return self.cfg.theta_week_present @ np.array([1.0, perceived_utility])

    def gen_week_present(self, perceived_utility, week_idx):
        mean = self.gen_week_present_mean(perceived_utility)
        noise = self._sample_noise(self.cfg.resid_week_present, week_idx)
        return np.clip(mean + noise, *self.cfg.limits_week_present)

    def gen_CAE_short_mean(self, CAE_avg):
        return self.cfg.theta_CAE_short @ np.array([1.0, CAE_avg])

    def gen_CAE_short(self, CAE_avg, week_idx):
        mean = self.gen_CAE_short_mean(CAE_avg)
        noise = self._sample_noise(
            self.cfg.resid_CAE_short, week_idx, self.cfg.resid_obs_CAE_short
        )
        return np.clip(mean + noise, *self.cfg.limits_CAE_short)


# quick smoke test
_env = Env(_cfg)
print("Env created OK")

Env created OK


In [4]:
def make_initial_state():
    """Return a state dict initialised at population-mean (0 in normalised scale)."""
    return {
        # decision-level lags
        "fourSC_lag1": 0.0,
        "hourly_pageview_lag1": 0.0,
        "prior2hour_step": 0.0,
        # daily lags (from yesterday)
        "yesterday_step": 0.0,
        "yesterday_fitbitwearing": 0.0,
        "yesterday_present": 0.0,
        "yesterday_pageview_morning": 0.0,
        "yesterday_pageview_afternoon": 0.0,
        # running 7-day averages
        "seven_day_step_avg": 0.0,
        "seven_day_pageview": 0.0,
        "past7days_daywearing": 0.0,
        # today's daily variables (set at start of each day)
        "fitbitwearing_morning": 0.0,
        "daily_present": 0.0,
        # weekly lags
        "CAE_avg_lastweek": 0.0,
        "perceived_utility_lastweek": 0.0,
        # anticipated affect
        "anticipated_affect_yesterday": 0.0,
        "anticipated_affect_lag1": 0.0,
        # external context -- defaults to normalised mean (0);
        # the caller may override any of these per step.
        "Previous7DaysRPA": 0.0,
        "recent_burden": 0.0,
        "recorded_physical_activity_lag1": 0.0,
        "yesterday_planning_prompt": 0.0,
        "yesterday_SalienceMessage": 0.0,
        "Interacted_7d_walk": 0.0,
        "Interacted_7d_salience": 0.0,
        # time features (updated each step)
        "is_weekend": 0.0,
        "dow_norm": 0.0,
        "dow_norm_lag1": 0.0,
        "decision_time": 0.0,
    }


def simulate(env, policy, nweek=None, Iw_schedule=None, seed=None,
             start_dow=1):
    """Run a full simulation of the vanilla testbed.

    Parameters
    ----------
    env : Env
        Environment object (wraps one participant's fitted parameters).
    policy : callable
        ``policy(state) -> action``  where action is 0 or 1 (WalkingSuggestion).
    nweek : int, optional
        Number of weeks to simulate (default: env.cfg.nweek = 11).
    Iw_schedule : array-like of shape (nweek,), optional
        Per-week query indicator (0 or 1).  Default: all zeros.
    seed : int, optional
        Random seed for reproducibility.
    start_dow : int
        Day-of-week for day 0 (1 = Mon … 7 = Sun).  Default 1 (Monday).

    Returns
    -------
    data : dict
        Arrays of generated trajectories, keyed by variable name.
    """
    if seed is not None:
        rd.seed(seed)

    K = env.K   # 2
    W = env.W   # 7
    if nweek is None:
        nweek = env.cfg.nweek
    D = nweek * W
    T = D * K

    if Iw_schedule is None:
        Iw_schedule = np.zeros(nweek)

    # ---- storage ----
    fourSC_all    = np.zeros(T)
    pageview_all  = np.zeros(T)
    action_all    = np.zeros(T)
    antic_all     = np.zeros(D)
    fitbit_all    = np.zeros(D)
    daily_all     = np.zeros(D)
    CAE_all       = np.zeros(nweek)
    pu_all        = np.zeros(nweek)
    wp_all        = np.zeros(nweek)
    cae_short_all = np.zeros(nweek)

    # ---- buffers for running averages ----
    step_buf   = np.zeros(W)
    pv_buf     = np.zeros(W)
    wear_buf   = np.zeros(W)

    # ---- state ----
    s = make_initial_state()
    yesterday_morning_WS = 0.0   # track yesterday morning's WS for daily models

    step_idx = 0
    day_idx  = 0

    for w in range(nweek):
        Iw = Iw_schedule[w]
        week_norm = (w + 1) / env.cfg.nweek  # normalised week number

        foursc_wk = np.zeros(K * W)   # 14 decision slots in the week
        antic_wk  = np.zeros(W)       # anticipated affect per day
        pw_wk     = np.zeros(W)       # daily pageview proxy
        dw_wk     = np.zeros(W)       # daily wearing proxy
        dp_wk     = np.zeros(W)       # daily present

        for d_w in range(W):
            d = w * W + d_w
            dow = ((start_dow - 1 + d) % 7) + 1          # 1-7
            dow_norm = dow / 7.0
            is_weekend = 1.0 if dow >= 6 else 0.0

            s["is_weekend"] = is_weekend
            s["dow_norm"]   = dow_norm

            # ---- daily mediators (start of day) ----
            ws_m = yesterday_morning_WS   # yesterday morning WS
            ws_a = 0.0                    # afternoon slot always 0 (matches fit)

            fitbit = env.gen_fitbitwearing(s, ws_m, ws_a, Iw, day_idx)
            s["fitbitwearing_morning"] = fitbit

            daily_pres = env.gen_dailysurvey(s, ws_m, ws_a, Iw, day_idx)
            s["daily_present"] = daily_pres

            antic = env.gen_antic(s, ws_m, ws_a, day_idx)

            fitbit_all[day_idx] = fitbit
            daily_all[day_idx]  = daily_pres
            antic_all[day_idx]  = antic

            # ---- decision-level loop ----
            today_fourSC   = np.zeros(K)
            today_pageview = np.zeros(K)
            today_action   = np.zeros(K)

            for t in range(K):
                s["decision_time"] = float(t)
                Ah = policy(s)
                action_all[step_idx] = Ah
                today_action[t]      = Ah

                fourSC  = env.gen_fourSC(s, Ah, step_idx)
                pv      = env.gen_pageview(s, Ah, Iw, step_idx)

                fourSC_all[step_idx]  = fourSC
                pageview_all[step_idx] = pv
                today_fourSC[t]  = fourSC
                today_pageview[t] = pv
                foursc_wk[d_w * K + t] = fourSC

                # within-day state updates
                s["fourSC_lag1"]          = fourSC
                s["hourly_pageview_lag1"] = pv
                s["prior2hour_step"]     = fourSC

                step_idx += 1

            # ---- end-of-day state updates ----
            daily_mean_step = np.mean(today_fourSC)
            daily_mean_pv   = np.mean(today_pageview)

            s["yesterday_step"]              = daily_mean_step
            s["yesterday_fitbitwearing"]     = fitbit
            s["yesterday_present"]           = daily_pres
            s["yesterday_pageview_morning"]  = today_pageview[0]
            s["yesterday_pageview_afternoon"] = today_pageview[1]
            s["dow_norm_lag1"]               = dow_norm
            s["anticipated_affect_lag1"]     = s["anticipated_affect_yesterday"]
            s["anticipated_affect_yesterday"] = antic

            # update rolling 7-day buffers
            step_buf[d % W] = daily_mean_step
            pv_buf[d % W]   = daily_mean_pv
            wear_buf[d % W] = fitbit
            n_buf = min(d + 1, W)
            s["seven_day_step_avg"]    = np.sum(step_buf) / n_buf
            s["seven_day_pageview"]    = np.sum(pv_buf)   / n_buf
            s["past7days_daywearing"]  = np.sum(wear_buf) / n_buf

            yesterday_morning_WS = today_action[0]

            # weekly aggregation helpers
            antic_wk[d_w] = antic
            pw_wk[d_w] = daily_mean_pv
            dw_wk[d_w] = fitbit
            dp_wk[d_w] = daily_pres

            day_idx += 1

        # ---- end-of-week: generate weekly outcomes ----
        cae = env.gen_CAE(
            s["CAE_avg_lastweek"], week_norm, foursc_wk, antic_wk, w
        )
        pu = env.gen_perceived_utility(
            s["perceived_utility_lastweek"], week_norm, pw_wk, dw_wk, dp_wk, w
        )
        wp = env.gen_week_present(pu, w)
        cs = env.gen_CAE_short(cae, w)

        CAE_all[w]       = cae
        pu_all[w]        = pu
        wp_all[w]        = wp
        cae_short_all[w] = cs

        s["CAE_avg_lastweek"]          = cae
        s["perceived_utility_lastweek"] = pu

    return {
        "fourSC": fourSC_all,
        "pageview": pageview_all,
        "action": action_all,
        "anticipated_affect": antic_all,
        "fitbitwearing": fitbit_all,
        "dailysurvey": daily_all,
        "CAE": CAE_all,
        "perceived_utility": pu_all,
        "week_present": wp_all,
        "CAE_short_avg": cae_short_all,
    }

In [5]:
# --- example: simulate all users under a random policy ---

def random_policy(state, p=0.5):
    """Bernoulli(p) walking suggestion."""
    return int(rd.random() < p)


user_ids = np.loadtxt(PARAMS_DIR / "user_ids.txt", dtype=int)
results = {}

for uid in user_ids:
    cfg = EnvConfig(uid)
    env = Env(cfg, noise="sequential")
    data = simulate(env, random_policy, seed=2026)
    results[uid] = data
    print(f"User {uid}:  CAE = {data['CAE'].round(3)},  "
          f"PU = {data['perceived_utility'].round(3)}")


# --- quick summary across users ---
print("\n--- Mean weekly CAE across all users ---")
all_cae = np.stack([results[uid]["CAE"] for uid in user_ids])
print(f"  per-week mean: {all_cae.mean(axis=0).round(3)}")
print(f"  overall mean:  {all_cae.mean():.3f}")

User 118:  CAE = [-0.388  0.041  0.38   1.321  1.321  1.321  1.321  1.321  1.321  1.321
  1.321],  PU = [0.333 0.565 0.981 1.239 1.223 1.276 1.498 1.147 1.301 1.358 1.432]
User 141:  CAE = [0.423 0.501 1.097 0.155 0.674 0.643 0.58  1.315 0.423 0.813 0.787],  PU = [0.    0.587 1.155 1.444 2.14  2.633 3.263 3.066 2.773 3.249 3.669]
User 143:  CAE = [-0.805 -1.022  0.154 -0.644  0.315 -0.031  0.109  0.236  0.705  0.245
 -0.232],  PU = [0.488 0.892 1.143 1.341 1.699 1.544 1.801 1.639 2.062 2.085 1.91 ]
User 151:  CAE = [0.346 0.259 0.456 0.72  0.611 0.591 0.759 0.889 0.698 0.541 0.694],  PU = [0.564 0.6   0.933 1.007 0.785 1.306 1.321 1.491 1.585 2.046 1.881]
User 160:  CAE = [0.405 0.539 0.299 0.506 0.656 0.216 0.196 0.422 0.63  0.188 0.256],  PU = [0.404 0.828 1.022 1.168 1.281 1.293 1.371 1.452 1.413 1.506 1.063]
User 170:  CAE = [0.262 0.184 0.243 0.266 0.243 0.242 0.243 0.235 0.253 0.22  0.191],  PU = [0.658 1.094 1.655 1.762 1.744 1.431 1.986 2.311 2.576 2.479 1.899]
User 184:  CAE =

In [6]:
print(data)

{'fourSC': array([ 0.43885714, -0.42061143,  0.32707693, -0.23144727,  0.47535264,
       -0.18006168,  0.07194106, -0.04361124,  0.03089378, -0.66639139,
        0.43449175, -0.33510981, -0.3272157 , -0.90186383, -0.1202677 ,
        0.16684874,  0.22732959, -0.280041  ,  0.0812614 , -1.19956076,
        0.00543853,  0.26186361,  0.54205599, -0.48889769, -0.15903397,
       -0.45244663, -0.60737448,  0.36944222,  0.70898483, -0.48424523,
       -0.43962145,  0.00402583,  0.10405935, -0.11418083,  0.67159983,
       -0.20080358,  0.87422951, -0.48989071, -0.25657111, -0.00340286,
       -0.3376398 , -0.46810007, -0.79073314,  0.02736967,  0.59262175,
       -0.61138224,  0.43430992, -0.0427251 , -0.11671065, -0.70220231,
        0.16816395, -0.27106461,  0.1274513 , -0.13337803,  0.61085573,
       -0.6698065 , -0.06945641, -0.49943359,  0.11469053, -0.24793374,
        1.11253817, -1.24009132,  0.1211181 , -0.20768249,  0.34355147,
       -0.11648027,  0.67089772, -1.70593008,  0.6840